In [13]:
import openai

print(openai.__version__)

2.8.1


In [14]:
import os
import sys

os.chdir("..")
sys.path.append("src/")

In [15]:
from openai import OpenAI

model = "gpt-5.1"
client = OpenAI()

user_prompt = "Hi, my name is Maliheh"
response = client.responses.create(model=model, input=user_prompt)

print(response.output_text)

Hi Maliheh, nice to meet you!  

How can I help you today?


In [16]:
from pydantic import BaseModel


class EventExtractInfo(BaseModel):
    name: str
    date: str
    participants: list[str]


response = client.responses.parse(
    model=model,
    input=[
        {"role": "system", "content": "Extract the event information"},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=EventExtractInfo,
)

print(response.output_parsed)

name='science fair' date='Friday' participants=['Alice', 'Bob']


In [17]:
model = "gpt-4.1-mini"

response = client.responses.create(
    model=model,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "what's in this image?"},
                {
                    "type": "input_image",
                    "image_url": "https://www.eatingwell.com/thmb/8MXOea6bVolkuLNQ1HhNt1tryIE=/750x0/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/article_7866255_foods-you-should-eat-every-week-to-lose-weight_-04-d58e9c481bce4a29b47295baade4072d.jpg",
                },
            ],
        }
    ],
)

print(response.output_text)

The image shows a variety of foods arranged on a light blue surface. The items include:

- A head of purple cabbage
- A bowl of brown rice
- Two heads of bok choy
- A fillet of raw salmon on a plate
- A bowl of wheat grains
- An avocado (whole and halved)
- Two eggs
- A bowl of sauerkraut
- Two apples
- A bowl of kimchi
- Two squares of dark chocolate
- A plate of pistachios
- A small bowl of chia seeds
- A stalk of broccolini or baby broccoli

The foods appear to be healthy and nutrient-rich.


In [18]:
import base64

model = "gpt-4.1"


def encode_image_by_path(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


image_path = (
    "/Users/malihehnorouzi/Desktop/foods-you-should-eat-every-week-to-lose-weight.webp"
)
base64_image = encode_image_by_path(image_path)

response = client.responses.create(
    model=model,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "what's in this image?"},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }
    ],
)

print(response.output_text)

This image displays a variety of healthy foods arranged on a light blue surface. Here is what is visible:

- Red cabbage (cut in half)
- Baby bok choy
- Broccolini
- Brown rice (in a bowl)
- Salmon fillet (on a plate)
- Farro or another whole grain (in a bowl)
- Avocado (whole and halved)
- Eggs
- Sauerkraut (in a bowl)
- Kimchi (in a bowl)
- Apples
- Pistachios (on a plate)
- Dark chocolate (bars)
- Chia seeds (in a bowl)

These foods are known for their nutritional benefits and are commonly included in healthy diets.


In [19]:
# lets you convert files (like images) into Base64 text


def encode_image_by_path(image_path):
    # Step 1: Open the image as bytes (binary data)
    # rb: read binary (images are not text, they're raw bytes)
    image_file = open(image_path, "rb")

    # Step 2: Read all the bytes from the image
    image_bytes = image_file.read()

    # Step 3: Convert those bytes into Base64 bytes
    base64_bytes = base64.b64encode(image_bytes)

    # Step 4: Convert Base64 bytes into a normal string
    base64_string = base64_bytes.decode("utf-8")

    # Step 5: Return the final Base64 string
    return base64_string

In [20]:

raw_data = b"\x89PNG\r\n\x1a\n"  # This is how real image bytes often look
print("RAW BYTES:", raw_data)

# Step 1: Base64 encode the raw bytes
base64_bytes = base64.b64encode(raw_data)
print("BASE64 (bytes):", base64_bytes)

# Step 2: Convert Base64 bytes → string
base64_string = base64_bytes.decode("utf-8")
print("BASE64 (string):", base64_string)

RAW BYTES: b'\x89PNG\r\n\x1a\n'
BASE64 (bytes): b'iVBORw0KGgo='
BASE64 (string): iVBORw0KGgo=


In [21]:
from services.analysis.schemas import IngredientsResponse
from services.chat_gpt.gpt import ChatGPT
from services.image_processing import encode_image_by_path
from services.prompts import FoodImageAnalyzerPrompts

model = "gpt-4.1"
image_path = (
    "/Users/malihehnorouzi/Desktop/foods-you-should-eat-every-week-to-lose-weight.webp"
)
base64_image = encode_image_by_path(image_path)

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_base64_image(
    model=model,
    system_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_USER_PROMPT,
    base64_image=base64_image,
    response_format=IngredientsResponse,
)

print(response)

name='Assorted Healthy Foods' ingredients=[Ingredient(ingredient_name='Red Cabbage', portiont='1/4 head'), Ingredient(ingredient_name='Brown Rice', portiont='1/2 cup raw'), Ingredient(ingredient_name='Baby Bok Choy', portiont='3 heads'), Ingredient(ingredient_name='Salmon Fillet', portiont='1 fillet (about 4 oz)'), Ingredient(ingredient_name='Broccolini', portiont='4 spears'), Ingredient(ingredient_name='Avocado', portiont='1 whole'), Ingredient(ingredient_name='Eggs', portiont='3 whole'), Ingredient(ingredient_name='Sauerkraut', portiont='1/2 cup'), Ingredient(ingredient_name='Kimchi', portiont='1/2 cup'), Ingredient(ingredient_name='Apples', portiont='2 whole'), Ingredient(ingredient_name='Dark Chocolate', portiont='4 squares'), Ingredient(ingredient_name='Pistachios', portiont='1/3 cup'), Ingredient(ingredient_name='Chia Seeds', portiont='2 tablespoons'), Ingredient(ingredient_name='Farro or Whole Grain', portiont='1/2 cup dry')]


In [22]:
from services.analysis.schemas import IngredientsResponse
from services.chat_gpt.gpt import ChatGPT
from services.prompts import FoodImageAnalyzerPrompts

model = "gpt-4.1"
image_url = "https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg"

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_image_url(
    model=model,
    system_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_USER_PROMPT,
    image_url=image_url,
    response_format=IngredientsResponse,
)

print(response)

name='Vietnamese Hu Tieu Nam Vang (Phnom Penh Noodle Soup)' ingredients=[Ingredient(ingredient_name='Rice noodles', portiont='1 serving (base of the bowl)'), Ingredient(ingredient_name='Prawns/Shrimp', portiont='2 pieces, whole (with shell)'), Ingredient(ingredient_name='Quail eggs', portiont='3 pieces'), Ingredient(ingredient_name='Pork slices', portiont='3-4 pieces, thinly sliced'), Ingredient(ingredient_name='Broth (clear yet golden)', portiont='1 cup (fills the bowl)'), Ingredient(ingredient_name='Chopped green onions', portiont='1 tablespoon, sprinkled on top'), Ingredient(ingredient_name='Fresh cilantro', portiont='1 tablespoon, sprinkled on top'), Ingredient(ingredient_name='Sliced red chili peppers', portiont='1 tablespoon, sliced and layered on top'), Ingredient(ingredient_name='Fresh mint leaves/greens', portiont='2 sprigs')]


In [23]:
from pydantic import BaseModel

from services.analysis.schemas import IngredientsResponse
from services.chat_gpt.gpt import ChatGPT
from services.image_processing import encode_image_by_url
from services.prompts import FoodImageAnalyzerPrompts

model = "gpt-4.1"
image_url = "https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg"
base64_image = encode_image_by_url(image_url)

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_base64_image(
    model=model,
    system_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_USER_PROMPT,
    base64_image=base64_image,
    response_format=IngredientsResponse,
)

print(response)

name='Seafood Noodle Soup (possibly Hu Tieu Nam Vang or similar Asian noodle soup)' ingredients=[Ingredient(ingredient_name='Rice noodles', portiont='1 bowl'), Ingredient(ingredient_name='Shrimp', portiont='2 whole pieces'), Ingredient(ingredient_name='Quail eggs', portiont='3 pieces'), Ingredient(ingredient_name='Pork (sliced)', portiont='3-4 slices'), Ingredient(ingredient_name='Fresh herbs (mint, cilantro, etc.)', portiont='A small handful'), Ingredient(ingredient_name='Chopped scallions', portiont='1 tablespoon'), Ingredient(ingredient_name='Red chili pepper slices', portiont='1 teaspoon'), Ingredient(ingredient_name='Broth (savory, possibly pork or seafood based)', portiont='Enough to cover ingredients')]


In [24]:
from services.analysis.ingredients import IngredientsAnalyzer

ingredients_analyzer = IngredientsAnalyzer()

image_url = "https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg"
result = ingredients_analyzer.analyze(image_url)
print(result)

name='Vietnamese Hu Tieu Nam Vang Noodle Soup' ingredients=[Ingredient(ingredient_name='Rice noodles', portiont='1 cup, cooked'), Ingredient(ingredient_name='Shrimp', portiont='2 pieces, large'), Ingredient(ingredient_name='Quail eggs', portiont='3 eggs'), Ingredient(ingredient_name='Pork slices', portiont='4-5 slices'), Ingredient(ingredient_name='Chopped green onion', portiont='1 tablespoon'), Ingredient(ingredient_name='Cilantro', portiont='1 tablespoon'), Ingredient(ingredient_name='Fresh red chili', portiont='1 chili, sliced'), Ingredient(ingredient_name='Broth (pork and shrimp based)', portiont='1 cup'), Ingredient(ingredient_name='Mint leaves', portiont='5-6 leaves')]


In [25]:
from services.analysis.nutrients import NutrientsAnalyzer
from services.analysis.schemas import Ingredient

nutrients_analyzer = NutrientsAnalyzer()

ingredients = [
    Ingredient(ingredient_name="Rice noodles", portiont="1 serving"),
    Ingredient(ingredient_name="Shrimp", portiont="2 pieces"),
    Ingredient(ingredient_name="Quail eggs", portiont="4 pieces"),
    Ingredient(ingredient_name="Pork slices", portiont="4-5 slices"),
    Ingredient(ingredient_name="Fresh herbs (such as mint)", portiont="A handful"),
    Ingredient(ingredient_name="Green onions", portiont="1 tablespoon, chopped"),
    Ingredient(ingredient_name="Red chili pepper", portiont="1 tablespoon, sliced"),
    Ingredient(ingredient_name="Broth (pork or seafood-based)", portiont="1 cup"),
]
result = nutrients_analyzer.analyze(ingredients)
print(result)

total_calories=420 total_protein_g=23.0 total_carbohydrates_g=56.0 total_fats_g=10.0 total_fiber_g=3.0
